In [42]:
from pyspark.sql import *
from pyspark.sql.functions import *

In [43]:
customers = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .load("/user/student/ecommerce/staging_zone/customers.csv")

customers.printSchema()
customers.show(5)

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: string (nullable = true)

+-----------+----------+---------+--------------------+------------+----------------+-----------+
|customer_id|first_name|last_name|               email|     country|            city|signup_date|
+-----------+----------+---------+--------------------+------------+----------------+-----------+
|          1|     Julia|   Morris|julia.morris1@exa...|      Jordan|       Burnsfort| 2026-08-08|
|          2|     Linda|Rodriguez|linda.rodriguez2@...|      Kuwait|Lake Lesliemouth| 2023-03-01|
|          3|   Heather|  Collins|heather.collins3@...|Saudi Arabia|   Davilaborough| 2022-01-25|
|          4|    Evelyn|   Weaver|evelyn.weaver4@ex...|       Qatar|     New Richard| 2023-01-25|
|          5|    Kendra|  

In [44]:
products = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .load("/user/student/ecommerce/staging_zone/products.csv")

products.printSchema()
products.show(5)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- stock_quantity: integer (nullable = true)

+----------+--------------------+-----------+-------+--------------+
|product_id|        product_name|   category|  price|stock_quantity|
+----------+--------------------+-----------+-------+--------------+
|         1|Front-line client...|   Clothing|1187.44|           968|
|         2|Configurable loca...|Accessories|1032.38|           908|
|         3|Sharable mobile t...|     Sports|1296.78|            98|
|         4|Centralized bifur...|  Furniture|  65.33|           621|
|         5|User-friendly asy...|      Shoes|1145.05|            26|
+----------+--------------------+-----------+-------+--------------+
only showing top 5 rows



In [45]:
orders = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .load("/user/student/ecommerce/staging_zone/orders.csv")

orders.printSchema()
orders.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- shipping_country: string (nullable = true)

+--------+-----------+--------------------+---------+----------------+
|order_id|customer_id|          order_date|   status|shipping_country|
+--------+-----------+--------------------+---------+----------------+
|       1|      19599|2025-10-20 11:02:...|completed|           Egypt|
|       2|      19317|2025-08-12 18:26:...|  pending|          Kuwait|
|       3|      57859|2025-10-02 19:42:...|cancelled|    Saudi Arabia|
|       4|      64568|2024-10-01 22:55:...|cancelled|           Qatar|
|       5|      84679|2025-05-02 22:00:...|completed|           Egypt|
+--------+-----------+--------------------+---------+----------------+
only showing top 5 rows



In [46]:
order_items = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .load("/user/student/ecommerce/staging_zone/order_items.csv")

order_items.printSchema()
order_items.show(5)

root
 |-- order_item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)

+-------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|quantity|unit_price|
+-------------+--------+----------+--------+----------+
|            1|  359084|     16071|       4|    115.32|
|            2|  417538|      9821|       4|   1578.51|
|            3|  298447|     15262|       2|    634.09|
|            4|  308534|      1107|       1|    1721.2|
|            5|  194989|     15802|       3|   1258.99|
+-------------+--------+----------+--------+----------+
only showing top 5 rows



In [47]:
customers = customers.dropDuplicates()
customers = customers.withColumn("signup_date", to_date(col("signup_date"), "yyyy-MM-dd"))

customers.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [48]:
products = products.dropDuplicates()
products = products.filter(
    (col("price").isNotNull()) & (col("price") > 0) & 
    (col("stock_quantity").isNotNull()) & (col("stock_quantity") >= 0)
)

In [49]:
orders = orders.dropDuplicates()
orders = orders.withColumn("order_date", to_timestamp(col("order_date"), "yyyy-MM-dd HH:mm:ss.S"))
orders = orders.withColumn("order_date", to_date(substring("order_date", 1, 10), "yyyy-MM-dd"))

orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- shipping_country: string (nullable = true)



In [50]:
order_items = order_items.dropDuplicates()
order_items = order_items.filter(
    (col("quantity").isNotNull()) & (col("quantity") > 0) & 
    (col("unit_price").isNotNull()) & (col("unit_price") >= 0)
)

In [55]:
sales = orders.join(order_items, "order_id", "inner") \
    .join(products, "product_id", "inner") \
    .join(customers,"customer_id", "inner")

In [56]:
sales = sales.withColumn("sales_amount", col("quantity") * col("unit_price"))

In [57]:
sales.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_amount: double (nullable = true)



In [58]:
total_sales = sales.agg(
    sum("sales_amount").alias("total_sales")
)

total_sales.show()

+-------------------+
|        total_sales|
+-------------------+
|6.026882991339996E9|
+-------------------+



In [59]:
total_quantity_sold = sales.agg(
    sum("quantity").alias("total_quantity_sold")
)

total_quantity_sold.show()

+-------------------+
|total_quantity_sold|
+-------------------+
|            6001251|
+-------------------+



In [61]:
order_totals = sales.groupBy("order_id") \
    .agg(sum("sales_amount").alias("order_total"))

avg_order_value = order_totals.agg(
    avg("order_total").alias("avg_order_value")
)

avg_order_value.show()

+------------------+
|   avg_order_value|
+------------------+
|12277.636404886702|
+------------------+



In [62]:
number_of_orders = sales.select(
    countDistinct("order_id").alias("number_of_orders")
)

number_of_orders.show()

+----------------+
|number_of_orders|
+----------------+
|          490883|
+----------------+



In [63]:
number_of_customers = sales.select(
    countDistinct("customer_id").alias("number_of_customers")
)

number_of_customers.show()

+-------------------+
|number_of_customers|
+-------------------+
|              99232|
+-------------------+



In [64]:
sales_by_country = sales.groupBy("country") \
    .agg(sum("sales_amount").alias("total_sales")) \
    .orderBy(desc("total_sales"))

sales_by_country.show()

+------------+--------------------+
|     country|         total_sales|
+------------+--------------------+
|       Egypt|     1.01824409189E9|
|         UAE|1.0113604628000003E9|
|Saudi Arabia|1.0077514019000001E9|
|      Jordan|1.0077170164500005E9|
|      Kuwait| 9.915593774700003E8|
|       Qatar| 9.902506408299999E8|
+------------+--------------------+



In [65]:
sales_by_category = sales.groupBy("category") \
    .agg(sum("sales_amount").alias("total_sales")) \
    .orderBy(desc("total_sales"))

sales_by_category.show()

+-----------+-------------------+
|   category|        total_sales|
+-----------+-------------------+
|   Clothing|     6.2317556855E8|
|       Home|6.224572203699999E8|
|      Shoes|6.160828442200001E8|
|Electronics|6.131962863400002E8|
|      Books|6.003464328200002E8|
|Accessories|5.952419449999998E8|
|  Furniture|5.936714337299993E8|
|     Sports|5.902372777500001E8|
|     Beauty|5.881343664700001E8|
|     Gaming|5.843396160900003E8|
+-----------+-------------------+



In [67]:
sales_by_day = sales.groupBy("order_date") \
    .agg(sum("sales_amount").alias("total_sales")) \
    .orderBy("order_date")

sales_by_day.show()

+----------+------------------+
|order_date|       total_sales|
+----------+------------------+
|2024-09-07|3874759.9699999993|
|2024-09-08| 8398860.120000001|
|2024-09-09|        8141191.18|
|2024-09-10| 8253913.490000001|
|2024-09-11| 7913380.210000002|
|2024-09-12| 7851271.389999994|
|2024-09-13| 8476425.749999996|
|2024-09-14|        8682196.21|
|2024-09-15| 8448867.639999995|
|2024-09-16| 8221108.719999995|
|2024-09-17|        8651925.86|
|2024-09-18|8255038.5299999975|
|2024-09-19|8087594.8999999985|
|2024-09-20| 9039002.789999997|
|2024-09-21| 8469588.079999998|
|2024-09-22| 8246190.069999999|
|2024-09-23|        8159726.35|
|2024-09-24|        8993464.88|
|2024-09-25| 7662265.170000001|
|2024-09-26| 7817639.209999998|
+----------+------------------+
only showing top 20 rows



In [76]:
sales_by_month = sales.withColumn("month", month("order_date")) \
    .withColumn("year", year("order_date")) \
    .groupBy("month", "year") \
    .agg(sum("sales_amount").alias("total_sales")) \
    .orderBy("year", "month")

sales_by_month.show(10)

+-----+----+--------------------+
|month|year|         total_sales|
+-----+----+--------------------+
|    9|2024|1.9331199270000017E8|
|   10|2024|2.5422639881000003E8|
|   11|2024|2.4948408238999984E8|
|   12|2024|2.5577841497999996E8|
|    1|2025|2.5591715853999996E8|
|    2|2025|2.2934754572000015E8|
|    3|2025| 2.560887445300001E8|
|    4|2025|2.4876624536000004E8|
|    5|2025|2.5521503787000018E8|
|    6|2025|2.5071554162000012E8|
+-----+----+--------------------+
only showing top 10 rows



In [78]:
sales_by_year = sales.withColumn("year", year("order_date")) \
    .groupBy("year") \
    .agg(sum("sales_amount").alias("total_sales")) \
    .orderBy("year")

sales_by_year.show()

+----+--------------------+
|year|         total_sales|
+----+--------------------+
|2024| 9.528008888800011E8|
|2025|3.0177951298799996E9|
|2026|2.0562869725800004E9|
+----+--------------------+



In [83]:
fact_sales = sales.select(
    "order_id", "customer_id", "product_id", "order_date", "quantity",
    "unit_price", col("sales_amount").alias("total_amount"),
    "category", "country", "status"
)

fact_sales.show(5)

+--------+-----------+----------+----------+--------+----------+------------------+-----------+-------+---------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|      total_amount|   category|country|   status|
+--------+-----------+----------+----------+--------+----------+------------------+-----------+-------+---------+
|     148|      10063|      5760|2025-06-13|       5|   1699.61|           8498.05|Electronics|  Egypt|completed|
|     148|      10063|     15560|2025-06-13|       5|   1302.66|            6513.3|Electronics|  Egypt|completed|
|     148|      10063|     19309|2025-06-13|       3|    1273.4|3820.2000000000003|   Clothing|  Egypt|completed|
|     148|      10063|      4398|2025-06-13|       4|   1909.62|           7638.48|Accessories|  Egypt|completed|
|     148|      10063|     15315|2025-06-13|       3|   1137.62|3412.8599999999997|  Furniture|  Egypt|completed|
+--------+-----------+----------+----------+--------+----------+------------------+-----

In [88]:
daily_order_totals =sales.withColumn("year", year("order_date")) \
    .withColumn("month", month("order_date")) \
    .withColumn("day", dayofmonth("order_date")) \
    .groupBy("year", "month", "day", "order_id") \
    .agg(
        sum("quantity").alias("order_quantity"),
        sum("sales_amount").alias("order_total")
)

daily_sales = daily_order_totals.groupBy("year", "month", "day") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("order_quantity").alias("total_quantity"),
        sum("order_total").alias("total_sales"),
        avg("order_total").alias("avg_order_value")
    ) \
    .orderBy("year", "month", "day")

daily_sales.show(5)

+----+-----+---+------------+--------------+-----------------+------------------+
|year|month|day|total_orders|total_quantity|      total_sales|   avg_order_value|
+----+-----+---+------------+--------------+-----------------+------------------+
|2024|    9|  7|         314|          3813|3874759.970000002|12339.999904458606|
|2024|    9|  8|         678|          8286|8398860.120000003|12387.699292035402|
|2024|    9|  9|         677|          8150|8141191.180000003|12025.393175775485|
|2024|    9| 10|         662|          8204|       8253913.49|12468.147265861027|
|2024|    9| 11|         663|          7835|7913380.209999998|11935.716757164402|
+----+-----+---+------------+--------------+-----------------+------------------+
only showing top 5 rows



In [90]:
customer_sales = sales.withColumn("customer_name", concat_ws(" ", col("first_name"), col("last_name"))) \
    .groupBy("customer_id", "customer_name", "country") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum("sales_amount").alias("total_spending")
    ) \
    .orderBy(col("total_spending").desc())

customer_sales.show(5)

+-----------+----------------+------------+------------+--------------+------------------+
|customer_id|   customer_name|     country|total_orders|total_quantity|    total_spending|
+-----------+----------------+------------+------------+--------------+------------------+
|      60985|       Ryan Hall|      Jordan|          13|           210|         242648.48|
|      70998|  Michael Wilson|      Jordan|          13|           217|         241643.88|
|      62026|     Chase Brown|Saudi Arabia|          14|           207|         241031.72|
|      47789|    Emily Thomas|      Kuwait|          15|           229|239637.10999999996|
|      81473|Jessica Crawford|       Qatar|          13|           200|229712.86000000002|
+-----------+----------------+------------+------------+--------------+------------------+
only showing top 5 rows



In [92]:
product_sales = sales.groupBy("product_id", "product_name", "category") \
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("sales_amount").alias("total_sales")
    ) \
    .orderBy(col("total_sales").desc())

product_sales.show(5)

+----------+--------------------+-----------+--------------+-----------------+
|product_id|        product_name|   category|total_quantity|      total_sales|
+----------+--------------------+-----------+--------------+-----------------+
|     11771|Reduced web-enabl...|      Shoes|           407|477799.4599999999|
|     12608|Integrated homoge...|   Clothing|           421|473207.6299999998|
|      1912|Balanced 5thgener...|Accessories|           431|        466242.88|
|       232|Up-sized human-re...|       Home|           426|        463075.84|
|     17293|Fully-configurabl...|     Beauty|           416|446674.0600000002|
+----------+--------------------+-----------+--------------+-----------------+
only showing top 5 rows



In [93]:
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_dw")

2026-09-09 11:29:08,391 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-09 11:29:08,394 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist
2026-09-09 11:29:16,722 WARN metastore.ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
2026-09-09 11:29:16,761 WARN metastore.ObjectStore: Failed to get database ecommerce_dw, returning NoSuchObjectException


DataFrame[]

In [94]:
spark.sql("SHOW DATABASES").show()

+------------+
|   namespace|
+------------+
|     default|
|   ecommerce|
|ecommerce_dw|
|   retail_db|
+------------+



In [95]:
fact_sales.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .saveAsTable("ecommerce_dw.fact_sales")

2026-09-09 11:36:12,018 WARN session.SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
2026-09-09 11:36:12,193 WARN conf.HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
2026-09-09 11:36:12,193 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-09 11:36:12,194 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist


In [96]:
daily_sales.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .saveAsTable("ecommerce_dw.daily_sales")

In [97]:
customer_sales.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .saveAsTable("ecommerce_dw.customer_sales")

In [98]:
product_sales.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .saveAsTable("ecommerce_dw.product_sales")